In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.datasets as datasets
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, TensorDataset
from torch.utils.tensorboard import SummaryWriter
from tqdm import tqdm
import torch.nn.functional as F

2025-05-01 13:40:52.226671: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-01 13:40:52.234749: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 13:40:52.300392: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-05-01 13:40:52.350104: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1746124852.399083  110425 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1746124852.41

In [ ]:
class DcGenerator(nn.Module):
    def __init__(self, ngf, out_size):
        super(DcGenerator, self).__init__()

        #assert(ngf % 32 == 0 and ngf >= 32)

        self.gen = nn.Sequential(
            self._block(1, ngf, 4, 1, 0, 1),
            self._block(ngf, ngf // 2, 4, 2, 0, 1),
            self._block(ngf // 2, ngf // 4, 4, 2, 0, 1),
            self._block(ngf // 4, ngf // 8, 4, 2, 0, 1),
            self._block(ngf // 8, 1, 10, 2, 0, 1),
            nn.Linear(16084, out_size),
            nn.Tanh(),
        )
    def _block(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
        return nn.Sequential(
            nn.ConvTranspose1d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
        )

    def forward(self, x):
        return self.gen(x).squeeze()
    
class DcDiscriminator(nn.Module):
    def __init__(self, ndf, dropout=0.2):
        super(DcDiscriminator, self).__init__()

        self.dropout = dropout

        self.disc = nn.Sequential(
            self._block(1, ndf // 16, 4, 1, 0, 1),
            self._block(ndf // 16, ndf // 8, 4, 2, 0, 1),
            self._block(ndf // 8, ndf // 4, 4, 2, 0, 1),
            self._block(ndf // 4, ndf // 2, 4, 2, 0, 1),
            self._block(ndf // 2, ndf, 4, 2, 0, 1),
            self._block(ndf, 1, 4, 2, 0, 1),
            nn.Linear(10, 1)
        )
    def _block(self, in_channels, out_channels, kernel_size, stride, padding, dilation):
        return nn.Sequential(
            nn.Conv1d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm1d(out_channels),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(self.dropout),
        )
    def forward(self, x):
        return self.disc(x).squeeze()
    
def test():
    noise_dim, N, feature_size = 1000, 5, 400

    x = torch.randn((N, 1, feature_size))
    disc = DcDiscriminator(512)
    print(disc(x).shape)
    assert disc(x).shape == (N, ), "Discriminator test failed"

    gen = DcGenerator(512, feature_size)
    z = torch.randn((N, 1, noise_dim))
    print(gen(z).shape)
    assert gen(z).shape == (N, feature_size), "Generator test failed"

test()

torch.Size([5])


RuntimeError: mat1 and mat2 shapes cannot be multiplied (5x1684 and 16084x400)